# Notebook 02 - Fintech Archive Restore and Session Readiness

This notebook is designed to run in its own Colab runtime. Local `/content` state from Notebook 00/01 may not exist when Notebook 02 starts.

Notebook 02 initializes a local Fintech project/session workspace under `/content`, restores archived/backfilled curated data from a Google Drive backup pack into that local workspace, then validates restored readiness. Google Drive is persistent archive/session backup storage only; it is not the active app workspace.

Confirmed Colab restore pattern:
- Initialize the local project/session workspace with native `fintech-init-project`.
- Use `fintech-backup-data restore` to restore backup-pack data into `/content/.../data/curated`.
- Keep `OVERWRITE_POLICY = "fail"` unless `replace` or `merge` is chosen intentionally.

Upstream backup-pack source layout is session-scoped:
`fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

Backfilled market data can be slow to move through Drive as many small Parquet files. Prefer packaged/sharded backup-pack restore into `/content` when available. Notebook code should orchestrate native upstream commands and validate paths; it must not reimplement archive, restore, session persistence, ingestion, or generated-data logic.


## 1. Optional Colab Runtime Package Setup

Install or expose upstream apps only in a live Colab runtime. Do not run package installation during repository validation.

In [ ]:
# Manual Colab-only setup cell.
# Uncomment only when the Colab runtime needs the upstream package.
# !python -m pip install --upgrade pip
# !python -m pip install "pandas-market-calendars>=5.0"
# !python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion

print("Package setup is manual Colab-only. Keep installs out of repository validation.")

## 2. Optional Google Drive Mount

Mount Drive only in Colab when you intentionally need access to archive/session backup storage. Drive mount is manual and must not run in local repository validation.

In [ ]:
# Manual Colab-only Drive mount cell.
# from google.colab import drive
# drive.mount("/content/drive")

print("Drive mount is manual Colab-only. Active app work remains under /content.")

## 3. Configure Backup-Pack Source and Local Restore Target

Set a real Drive folder, session id, and backup id before any restore preview or live restore. These placeholders are intentionally invalid until replaced in a live runtime.

Notebook 02 consumes a session backup-pack source produced by Notebook 01 or another prior workflow while local backfill state existed. It does not create the backup source.

Expected source shape:
- `fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

The local restore target remains under `/content`; Drive is only the backup-pack source.


In [ ]:
from pathlib import Path

DRIVE_FOLDER_NAME = "REPLACE_WITH_DRIVE_FOLDER_NAME"
SESSION_ID = "REPLACE_WITH_SESSION_ID"
BACKUP_ID = "REPLACE_WITH_BACKUP_ID"
SESSION_NAME = "extraction_daily_bars_demo"

DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
DRIVE_PROJECT_ROOT = DRIVE_ROOT / "fintech-market-ingestion"
DRIVE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / SESSION_ID
DRIVE_BACKUP_ROOT = DRIVE_SESSION_ROOT / "backups" / BACKUP_ID
DRIVE_BACKUP_MANIFEST = DRIVE_BACKUP_ROOT / "manifest.json"

FINTECH_ROOT = Path("/content/fintech-market-ingestion-demo")
RESTORE_ROOT = FINTECH_ROOT / "data" / "curated"
OVERWRITE_POLICY = "fail"
ARCHIVE_RESTORE_COMMAND_CANDIDATE = "fintech-backup-data"
VALID_OVERWRITE_POLICIES = {"fail", "replace", "merge"}

if not str(FINTECH_ROOT).startswith("/content/"):
    raise ValueError(f"Active Fintech workspace must stay under /content: {FINTECH_ROOT}")

if not str(RESTORE_ROOT).startswith("/content/"):
    raise ValueError(f"Archive restore root must stay under /content: {RESTORE_ROOT}")

print("Drive project root:", DRIVE_PROJECT_ROOT)
print("Drive session root:", DRIVE_SESSION_ROOT)
print("Drive backup/archive source root:", DRIVE_BACKUP_ROOT)
print("Drive backup manifest:", DRIVE_BACKUP_MANIFEST)
print("Local restore target workspace:", FINTECH_ROOT)
print("Archive restore root:", RESTORE_ROOT)
print("Overwrite policy:", OVERWRITE_POLICY)
print("Session name:", SESSION_NAME)


## 4. Confirm Native Command Availability

These checks report whether expected upstream command entry points are visible in the current runtime. `fintech-init-project` initializes the local `/content` workspace/session, and `fintech-backup-data restore` is the confirmed archive/backfilled data restore command. A dedicated session-file restore command may exist upstream, but it is not the primary archive data restore path for Notebook 02.


In [ ]:
import shutil

COMMAND_CANDIDATES = [
    "fintech-init-project",
    "fintech-backup-data",
]

COMMAND_AVAILABLE = {name: shutil.which(name) is not None for name in COMMAND_CANDIDATES}
for name, available in COMMAND_AVAILABLE.items():
    print(f"{name}: {'available' if available else 'missing'}")

if not COMMAND_AVAILABLE.get("fintech-init-project", False):
    print("BLOCKED: fintech-init-project is not visible. Confirm upstream CLI installation before initialization.")

if not COMMAND_AVAILABLE.get(ARCHIVE_RESTORE_COMMAND_CANDIDATE, False):
    print("BLOCKED: fintech-backup-data is not visible. Confirm upstream CLI installation before archive restore.")


## 5. Help Surfaces for Initialization and Backup-Pack Restore

Run help commands before live initialization or restore. Missing local commands are acceptable in repository validation when configured as expected warnings.

Confirmed findings from live Colab testing:
- `fintech-init-project --notebooks` uses `--notebooks` as a standalone flag.
- `fintech-backup-data restore` accepts `--backup-pack-dir`, `--restore-root`, and `--overwrite-policy fail|replace|merge`.


In [ ]:
!fintech-init-project --help
!fintech-backup-data --help
!fintech-backup-data restore --help
!fintech-backup-data validate --help
!fintech-backup-data inspect --help


## 6. Initialize Local Restore Workspace

`/content` is the active local workspace. Google Drive is only the archive/session backup source. Restore should target an initialized Fintech project/session workspace, not an arbitrary empty path.

`--notebooks` is a standalone `fintech-init-project` flag. Do not pass a value after `--notebooks`. Confirm `fintech-init-project --help` before changing flags. `SESSION_NAME` may be changed intentionally for a new smoke run.

Initialization is manual Colab-only because it may create runtime directories under `/content`. Initialization must not run during repository validation.


In [ ]:
import shlex

INIT_PROJECT_COMMAND = [
    "fintech-init-project",
    "--root", str(FINTECH_ROOT),
    "--notebooks",
    "--with-session",
    "--session-name", SESSION_NAME,
]

print("Fintech project initialization command preview:")
print(" ".join(shlex.quote(part) for part in INIT_PROJECT_COMMAND))
print("Run this in Colab before restore if the local target workspace does not exist.")
print("Confirm fintech-init-project flags against fintech-init-project --help before enabling live execution.")


In [ ]:
# Manual Colab-only local restore workspace initialization.
# Confirm fintech-init-project flags before enabling.
# import subprocess
# subprocess.run(INIT_PROJECT_COMMAND, check=True)

print("Local restore workspace initialization remains manual Colab-only.")

## 7. Validate Local Restore Workspace Structure

Validate the initialized `/content` workspace before restore. The target must exist and include the expected Fintech project/session directories before archive restore preflight runs.

In [ ]:
EXPECTED_RESTORE_TARGET_PATHS = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data",
    FINTECH_ROOT / "data" / "curated",
]

RESTORE_TARGET_READY = all(path.exists() for path in EXPECTED_RESTORE_TARGET_PATHS)

for path in EXPECTED_RESTORE_TARGET_PATHS:
    if not path.exists():
        print(f"BLOCKED: Missing restore target path: {path}")

if RESTORE_TARGET_READY:
    print("Restore target workspace ready.")
else:
    print("Restore target workspace is not ready. Initialize the local workspace before restore.")

## 8. Archive Restore Preflight

Run this non-mutating preflight after workspace initialization and before restore previews or live restore. It blocks placeholder Drive/session/backup values, confirms Drive is mounted, checks backup-pack source paths, verifies local target readiness, and keeps restore output under `/content`.


In [ ]:
ARCHIVE_RESTORE_PREFLIGHT_READY = True
preflight_messages = []

DRIVE_MOUNT_ROOT = Path("/content/drive/MyDrive")

if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set DRIVE_FOLDER_NAME to an intentional Drive folder before restore actions.")

if SESSION_ID == "REPLACE_WITH_SESSION_ID":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set SESSION_ID to an intentional session id before restore actions.")

if BACKUP_ID == "REPLACE_WITH_BACKUP_ID":
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set BACKUP_ID to an intentional backup id before restore actions.")

if not DRIVE_MOUNT_ROOT.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Mount Google Drive before checking backup-pack source paths.")

if DRIVE_MOUNT_ROOT.exists() and not DRIVE_BACKUP_ROOT.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing backup-pack source path: {DRIVE_BACKUP_ROOT}")

if DRIVE_MOUNT_ROOT.exists() and not DRIVE_BACKUP_MANIFEST.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing backup-pack manifest path: {DRIVE_BACKUP_MANIFEST}")

if not str(FINTECH_ROOT).startswith("/content/"):
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Local restore target must stay under /content: {FINTECH_ROOT}")

if not str(RESTORE_ROOT).startswith("/content/"):
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Archive restore root must stay under /content: {RESTORE_ROOT}")

if not RESTORE_TARGET_READY:
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Initialize the local Fintech workspace before archive restore.")

if not RESTORE_ROOT.parent.exists():
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append(f"Missing restore root parent after initialization: {RESTORE_ROOT.parent}")

if OVERWRITE_POLICY not in VALID_OVERWRITE_POLICIES:
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("Set OVERWRITE_POLICY to fail, replace, or merge before live restore.")

if not COMMAND_AVAILABLE.get(ARCHIVE_RESTORE_COMMAND_CANDIDATE, False):
    ARCHIVE_RESTORE_PREFLIGHT_READY = False
    preflight_messages.append("fintech-backup-data is not available in this runtime.")

for message in preflight_messages:
    print("BLOCKED:", message)

if ARCHIVE_RESTORE_PREFLIGHT_READY:
    print("Archive restore preflight ready.")
else:
    print("Archive restore preflight is not ready. Fix blocked items before preview or live restore.")


## 9. Backup-Pack Restore Command Preview

Initialize the local restore target under `/content` first, then restore intentionally. The confirmed archive data restore command is `fintech-backup-data restore` with `--backup-pack-dir`, `--restore-root`, and `--overwrite-policy`.

Use the help cells above to review `fintech-backup-data validate` and `fintech-backup-data inspect` before live restore. The repository-side CLI parser keeps this cell focused on one restore preview so validation does not execute or conflate manual commands.

Start with `OVERWRITE_POLICY = "fail"`. Use `replace` or `merge` only intentionally. Drive is the backup-pack source only; do not use Drive as the active workspace.


In [ ]:
import shlex

ARCHIVE_RESTORE_COMMAND_TEXT = (
    "fintech-backup-data restore "
    f"--backup-pack-dir {shlex.quote(str(DRIVE_BACKUP_ROOT))} "
    f"--restore-root {shlex.quote(str(RESTORE_ROOT))} "
    f"--overwrite-policy {shlex.quote(OVERWRITE_POLICY)}"
)

ARCHIVE_RESTORE_COMMAND = shlex.split(ARCHIVE_RESTORE_COMMAND_TEXT)

print("Archive restore command preview:")
print(ARCHIVE_RESTORE_COMMAND_TEXT)

if not ARCHIVE_RESTORE_PREFLIGHT_READY:
    raise RuntimeError("Archive restore preflight is not ready. Do not run backup-pack restore yet.")

print("Preflight is ready. Keep live backup-pack restore manual Colab-only.")
# import subprocess
# subprocess.run(ARCHIVE_RESTORE_COMMAND, check=True)


## 10. Manual Live Archive Restore

Live restore is manual Colab-only. First initialize the local restore target under `/content`, then restore curated data into the local restore root.

Start with `OVERWRITE_POLICY = "fail"`. Use `replace` or `merge` only intentionally. Restore target is local `/content/.../data/curated`; Drive is backup-pack source only. Do not use Drive as the active workspace.


In [ ]:
# Manual Colab-only archive data restore.
# if not ARCHIVE_RESTORE_PREFLIGHT_READY:
#     raise RuntimeError("Archive restore preflight is not ready. Fix blocked items before live restore.")
#
# import subprocess
# subprocess.run(ARCHIVE_RESTORE_COMMAND, check=True)

print("Live archive restore remains commented/manual-only.")


## 11. Validate Restored Workspace Structure

After restore, validate expected local `/content` workspace paths. These checks are lightweight and should not print broad generated file listings.


In [ ]:
EXPECTED_RESTORED_PATHS = [
    FINTECH_ROOT,
    FINTECH_ROOT / "configs",
    FINTECH_ROOT / "reports",
    FINTECH_ROOT / "artifacts",
    FINTECH_ROOT / "data" / "curated",
]

for path in EXPECTED_RESTORED_PATHS:
    print(f"{path}: exists={path.exists()}")

## 12. Validate Restored Session Metadata

Review the presence of restored session manifests. Do not paste full manifest contents or generated payload listings into committed notebook source.


In [ ]:
import json

RESTORED_SESSION_MANIFESTS = sorted(
    (FINTECH_ROOT / "artifacts" / "sessions").glob("*/session_manifest.json"),
    key=lambda path: path.stat().st_mtime if path.exists() else 0,
)

print("Restored session manifest count:", len(RESTORED_SESSION_MANIFESTS))

if RESTORED_SESSION_MANIFESTS:
    latest_manifest_path = RESTORED_SESSION_MANIFESTS[-1]
    print("Latest restored session manifest:", latest_manifest_path)
    latest_manifest = json.loads(latest_manifest_path.read_text(encoding="utf-8"))
    safe_keys = ["session_id", "session_name", "created_at"]
    print("Safe restored session summary keys:", [key for key in safe_keys if key in latest_manifest])
else:
    print("No restored session manifest found. Rerun restore after confirming the archive/session source.")

## 13. Validate Restored Curated/Backfilled Data Presence

Check whether curated data appears to exist after restore without printing broad file listings. Prefer summary counts over path dumps.


In [ ]:
if RESTORE_ROOT.exists():
    RESTORED_CURATED_FILES = sorted(RESTORE_ROOT.rglob("*.parquet"))
    print("Archive restore root exists:", RESTORE_ROOT)
    print("Restored curated parquet file count:", len(RESTORED_CURATED_FILES))
else:
    print("Archive restore root is missing:", RESTORE_ROOT)


## 14. Runtime and Repository Boundaries

Keep active app work under `/content`. Keep Google Drive as archive/session backup storage only. Do not commit restored workspaces, archive packages, generated data, session payloads, restore outputs, notebook outputs, execution counts, credentials, private paths, or personal Drive folder names.

Notebook 02 consumes an existing archive/session backup. Archive creation, advanced archive shard/package inspection, archive transfer workflows, StratLake initialization, feature generation, strategy smoke tests, and backtest review remain deferred to Notebook 03+ or later notebooks.


## Notebook Summary

Notebook 02 initializes a local Fintech project/session workspace under `/content`, restores archived/backfilled curated data from a Drive backup pack with native `fintech-backup-data restore`, then checks workspace, session, and curated-data readiness. It is designed for separate Colab runtimes and does not assume Notebook 00/01 local `/content` state still exists.

Source layout reminder for restore inputs:
- `fintech-market-ingestion/sessions/<SESSION_ID>/backups/<BACKUP_ID>/manifest.json`

Notebook 02 now has two readiness sides before restore: source-side backup-pack presence in Drive and target-side local `/content` project/session initialization. The notebook should initialize the Fintech workspace under `/content` with `fintech-init-project --notebooks --with-session` rather than asking users to manually recreate directories.

Before committing source updates:

- Clear all outputs.
- Keep every code cell `execution_count` as `null`.
- Confirm no real session id, backup id, private local path, personal Drive path, credentials, archive payload, restore output, or generated data has been committed.
- Keep live initialization and restore manual Colab-only.
- Keep Notebook 03+ archive creation and downstream StratLake workflows deferred.
